#### NTU Quantum Practitioner Webinar Series

# Utility scale dynamic circuits
Boseong Kim (Apr 22, 2026) <br>
IBM Quantum

In this exercise, we will follow the dynamic circuit implementation presented at last year's QDC [[YouTube link]](https://youtu.be/rs3nlhRK7So?si=NAu71tuz8gZzIu59) with the latest qiskit runtime features.

In [ ]:
# Required packages for this notebook
# (For Mac users) Please add "" like: "qiskit-ibm-runtime[visualization]"

# !pip install -U qiskit-ibm-runtime[visualization] nbformat

In [ ]:
import qiskit

qiskit.__version__

In [ ]:
import qiskit_ibm_runtime

qiskit_ibm_runtime.__version__  # make sure that the version is above v0.43

In [ ]:
# Import basic classes
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, Sampler

### 0. AKLT state preparation in a quantum circuit

Here we build a unitary gate that prepares the AKLT state on a quantum circuit.

In [ ]:
# Build 3-qubit unitary

t1 = 2*np.arctan(-1/np.sqrt(2))
t2 = 2*np.arctan(np.sqrt(2))

U_qc = QuantumCircuit(3, name="U")
U_qc.cx(0, 1)
U_qc.cry(t1, 1, 0)
U_qc.x(1)
U_qc.cry(t2, 1, 2)
U_qc.x(1)
U_qc.cx(0, 1)
U_qc.cx(1, 2)
U_qc.cx(2, 1)
U_qc.draw("mpl")

### 1. Unitary implementation

The example of unitary AKLT state preparation is as below:

In [ ]:
# Unitary AKLT state example

q = QuantumRegister(14, name="q")
cr_edge = ClassicalRegister(2, name="cr_edge")
cr_AKLT = ClassicalRegister(12, name="cr_AKLT")

unitary = QuantumCircuit(q, cr_edge, cr_AKLT)
unitary.x([6, 7])
unitary.h(7)
unitary.cx(7, 6)
unitary.append(U_qc, [ 6, 5, 4])
unitary.append(U_qc, [ 4, 3, 2])
unitary.append(U_qc, [ 2, 1, 0])
unitary.append(U_qc, [ 7, 8, 9])
unitary.append(U_qc, [ 9,10,11])
unitary.append(U_qc, [11,12,13])
unitary.barrier()
unitary.measure([0, 13], cr_edge)
unitary.measure(q[1: 13], cr_AKLT)
unitary.draw("mpl")

In [ ]:
# Decomposed circuit for reference
unitary.decompose().draw("mpl", fold=-1)

#### Scheduled circuits

Here, you can meet the updated feature of Qiskit IBM Runtime: [scheduled circuits!](https://quantum.cloud.ibm.com/docs/en/guides/qiskit-runtime-circuit-timing)

In [ ]:
# Connect to the Qiskit IBM Runtime
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
sampler = Sampler(mode=backend)

# Access to the schedular timing data through the sampler primitive
sampler.options.experimental = {
    "execution": {
        "scheduler_timing": True,
    },
}

# Run sampler job
job_unitary = sampler.run([pm.run(unitary)])  # Estimated QPU time 2s

In [ ]:
# Extract scheduler timing data
timing_unitary = job_unitary.result()[0].metadata["compilation"]["scheduler_timing"]["timing"]

In [ ]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

# Visualize the circuit schedule
fig = draw_circuit_schedule_timing(
    circuit_schedule=timing_unitary,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)
fig.show(renderer="notebook")

You can drag around the circuit schedule to zoom in and out to the plot.

### 2. Measurement-based implementation without dynamic circuits

#### Exercise 1: Build the measurement-based AKLT state circuit without dynamic circuits

You can refer to the slide for the reference circuit implementation.

In [ ]:
# Exercise 1: Build the measurement-based AKLT state circuit without dynamic circuits
# Use below registers in your circuit
q = QuantumRegister(18, name="q")
cr_fusion = ClassicalRegister(4, name="cr_fusion")
cr_edge = ClassicalRegister(2, name="cr_edge")
cr_AKLT = ClassicalRegister(12, name="cr_AKLT")
mbpp = QuantumCircuit(q, cr_fusion, cr_edge, cr_AKLT)

# Build the quantum operations in the circuit
# You don't need to build the last circuit measurements
### Put your codes below ###


### Put your codes above ###

# The very last circuit measurement block
mbpp.barrier()
mbpp.measure([0, 17], cr_edge)
mbpp.measure(q[1:5] + q[7:11] + q[13:17], cr_AKLT)

# Draw the circuit
mbpp.draw("mpl", fold=-1)

In [ ]:
# Run the circuit and visualize the circuit timing
job_mbpp = sampler.run([pm.run(mbpp)])  # Estimated QPU time 2s
timing_mbpp = job_mbpp.result()[0].metadata["compilation"]["scheduler_timing"]["timing"]
fig = draw_circuit_schedule_timing(
    circuit_schedule=timing_mbpp,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)
fig.show(renderer="notebook")

### 3. Measurement-based implementation with dynamic circuits

Here I provided a reference quantum circuit implementation for the AKLT state preparation.

In [ ]:
# This is the example of dynamic quantum circuit
q = QuantumRegister(18, name="q")
cr_fusion = ClassicalRegister(4, name="cr_fusion")
cr_edge = ClassicalRegister(2, name="cr_edge")
cr_AKLT = ClassicalRegister(12, name="cr_AKLT")
mbff = QuantumCircuit(q, cr_fusion, cr_edge, cr_AKLT)

# Parallel unitary operation
mbff.x([2, 3, 8, 9, 14, 15])
mbff.h([3, 9, 15])
mbff.cx([3, 9, 15], [2, 8, 14])
mbff.append(U_qc, [ 2, 1, 0])
mbff.append(U_qc, [ 3, 4, 5])
mbff.append(U_qc, [ 8, 7, 6])
mbff.append(U_qc, [ 9,10,11])
mbff.append(U_qc, [14,13,12])
mbff.append(U_qc, [15,16,17])
mbff.barrier()

# Fusion between parallel AKLT states
# Bell measurement
mbff.cx([6, 12], [5, 11])
mbff.h([6, 12])
mbff.barrier()
mbff.measure([5,6, 11,12], cr_fusion)

# Classical feed-forward
with mbff.if_test((cr_fusion[0], 1)):
    mbff.swap(2, 1)
    mbff.swap(3, 4)
with mbff.if_test((cr_fusion[0], 1)):
    mbff.x(0)
with mbff.if_test((cr_fusion[2], 1)):
    mbff.swap(14,13)
    mbff.swap(15,16)
with mbff.if_test((cr_fusion[2], 1)):
    mbff.x(17)
mbff.barrier()
with mbff.if_test((cr_fusion[1], 1)):
    mbff.z([2, 1])
    mbff.z([3, 4])
with mbff.if_test((cr_fusion[1], 1)):
    mbff.z(0)
with mbff.if_test((cr_fusion[3], 1)):
    mbff.z([14, 13])
    mbff.z([15, 16])
with mbff.if_test((cr_fusion[3], 1)):
    mbff.z(17)

# The very last circuit measurement block
mbff.barrier()
mbff.measure([0, 17], cr_edge)
mbff.measure(q[1:5] + q[7:11] + q[13:17], cr_AKLT)

# Draw the circuit
mbff.draw("mpl", fold=-1)

In [ ]:
# Run the circuit and visualize the circuit timing
job_mbff = sampler.run([pm.run(mbff)])  # Estimated QPU time 2s
timing_mbff = job_mbff.result()[0].metadata["compilation"]["scheduler_timing"]["timing"]
fig = draw_circuit_schedule_timing(
    circuit_schedule=timing_mbff,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)
fig.show(renderer="notebook")

#### Exercise 2: Try implementing the dynamic decoupling on your own!

The code is basically the same with the previous circuit, so I copied the circuit from above. You now need to build a DD sequence based on the stretch method introduced in [this page](https://quantum.cloud.ibm.com/docs/en/guides/stretch).

In [ ]:
# Exercise 2: Try implementing the dynamic decoupling on your own
# Use below registers and stretches in your circuit
q = QuantumRegister(18, name="q")
cr_fusion = ClassicalRegister(4, name="cr_fusion")
cr_edge = ClassicalRegister(2, name="cr_edge")
cr_AKLT = ClassicalRegister(12, name="cr_AKLT")
mbff_dd = QuantumCircuit(q, cr_fusion, cr_edge, cr_AKLT)
s_list = [mbff_dd.add_stretch(f"s{i}") for i in range(14)]

# The code is the same one with the previous circuit
# Parallel unitary operation
mbff_dd.x([2, 3, 8, 9, 14, 15])
mbff_dd.h([3, 9, 15])
mbff_dd.cx([3, 9, 15], [2, 8, 14])
mbff_dd.append(U_qc, [2,1,0])
mbff_dd.append(U_qc, [3,4,5])
mbff_dd.append(U_qc, [8,7,6])
mbff_dd.append(U_qc, [9,10,11])
mbff_dd.append(U_qc, [14,13,12])
mbff_dd.append(U_qc, [15,16,17])
mbff_dd.barrier()

# Fusion between parallel AKLT states
# Bell measurement
mbff_dd.cx([6, 12], [5, 11])
mbff_dd.h([6, 12])
mbff_dd.barrier()
mbff_dd.measure([5,6, 11,12], cr_fusion)

# Here, add the dynamic decoupling sequence along with the measurement
### Put your codes below ###


### Put your codes above ###

# Classical feed-forward
with mbff_dd.if_test((cr_fusion[0], 1)):
    mbff_dd.swap(2, 1)
    mbff_dd.swap(3, 4)
with mbff_dd.if_test((cr_fusion[0], 1)):
    mbff_dd.x(0)
with mbff_dd.if_test((cr_fusion[2], 1)):
    mbff_dd.swap(14,13)
    mbff_dd.swap(15,16)
with mbff_dd.if_test((cr_fusion[2], 1)):
    mbff_dd.x(17)
mbff_dd.barrier()
with mbff_dd.if_test((cr_fusion[1], 1)):
    mbff_dd.z([2, 1])
    mbff_dd.z([3, 4])
with mbff_dd.if_test((cr_fusion[1], 1)):
    mbff_dd.z(0)
with mbff_dd.if_test((cr_fusion[3], 1)):
    mbff_dd.z([14, 13])
    mbff_dd.z([15, 16])
with mbff_dd.if_test((cr_fusion[3], 1)):
    mbff_dd.z(17)

# The very last circuit measurement block
mbff_dd.barrier()
mbff_dd.measure([0, 17], cr_edge)
mbff_dd.measure(q[1:5] + q[7:11] + q[13:17], cr_AKLT)

# Draw the circuit
mbff_dd.draw("mpl", fold=-1)

In [ ]:
# Run the circuit and visualize the circuit timing
job_mbff_dd = sampler.run([pm.run(mbff_dd)])  # Estimated QPU time 2s
timing_mbff_dd = job_mbff_dd.result()[0].metadata["compilation"]["scheduler_timing"]["timing"]
fig = draw_circuit_schedule_timing(
    circuit_schedule=timing_mbff_dd,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)
fig.show(renderer="notebook")

#### MidCircuitMeasure instruction

A `MidCircuitMeasure` instruction [[Release notes]](https://quantum.cloud.ibm.com/docs/en/guides/stretch) is an experimental feature in our latest devices. You may face several issues when using this experimental feature, but please don't hesitate to try it and give feedback to our [qiskit-ibm-runtime GitHub](https://github.com/Qiskit/qiskit-ibm-runtime/issues)!

### 4. Hybrid implementation
#### Exercise 3. Try building the hybrid circuit in the slide on your own!

You can refer to the slide for the reference circuit implementation.

In [ ]:
# Exercise 3: Try building the hybrid circuit in the slide on your own
# Use below registers in your circuit
q = QuantumRegister(16, name="q")
cr_fusion = ClassicalRegister(2, name="cr_fusion")
cr_edge = ClassicalRegister(2, name="cr_edge")
cr_AKLT = ClassicalRegister(12, name="cr_AKLT")
hybrid = QuantumCircuit(q, cr_fusion, cr_edge, cr_AKLT)

# Build the quantum operations in the circuit
# You don't need to build the last circuit measurements
### Put your codes below ###




### Put your codes above ###

# The very last circuit measurement block
hybrid.barrier()
hybrid.measure([0, 15], cr_edge)
hybrid.measure(q[1:6] + q[9:16], cr_AKLT)

# Draw the circuit
hybrid.draw("mpl", fold=-1)

In [ ]:
# Run the circuit and visualize the circuit timing
job_hybrid = sampler.run([pm.run(hybrid)])  # Estimated QPU time 2s
timing_hybrid = job_hybrid.result()[0].metadata["compilation"]["scheduler_timing"]["timing"]
fig = draw_circuit_schedule_timing(
    circuit_schedule=timing_hybrid,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)
fig.show(renderer="notebook")

© IBM Corp., 2017-2026